# Best On-Device SLM for GridRoute + MazeBench -- Kaggle Training Notebook

Runs the full pipeline from `neuro-symbolic-pathfinding` end to end on a free Kaggle GPU: Phase 1 baselines (Gemma 4 E2B/E4B + AlphaMaze on GridRoute 5x5 and MazeBench), a Phase 1b full-precision ablation (everything above runs 4-bit -- necessary for training, not for baseline inference, so this checks directly how much quality that costs rather than assuming), then Phase 2 training recipes on Gemma 4 E2B. See `idea.md`, `refined-idea.md`, and `experiment-plan.md` in the repo for the full plan this executes. No specific technique is the point here -- get Gemma 4 as good as possible at both benchmarks, report whichever recipe actually works.

**Built for unattended execution.** Every stage below runs through a `run_stage()` wrapper: it never raises, always logs success/failure/skip to `./results/stage_log.json`, and a later stage that depends on an earlier one's output (e.g. GRPO needing SFT's checkpoint) is skipped with a clear reason rather than crashing if that input is missing. Use **Save Version -> Save & Run All (Commit)** to run the whole thing in the background on Kaggle's servers -- close the tab, come back later, and check the final "Run Summary" cell plus `results.zip` rather than watching every cell live.

**Before running:**

1. Settings (right panel) -> Accelerator -> **GPU T4 x2** (or P100).
2. Settings -> **Internet: On** (needed for pip installs, HuggingFace downloads, and cloning the repo).
3. Give this notebook access to the repo (it's private). Two options:
   - **Recommended**: Add-ons -> Secrets -> add a secret named `GITHUB_TOKEN` with a GitHub [fine-grained personal access token](https://github.com/settings/tokens) scoped read-only to this one repo, **then toggle it ON for this specific notebook** (adding a secret to your account doesn't attach it automatically). Never paste the token directly into a cell or into chat -- Kaggle Secrets keeps it out of the notebook file entirely, and this notebook only ever reads it through `kaggle_secrets`. If a token is ever exposed anywhere else (a cell, an error message, a chat), revoke it on GitHub and issue a new one -- don't reuse it.
   - **Alternative**: zip the repo yourself, upload it as a private Kaggle Dataset, attach it to this notebook (Add data -> your dataset), and change `REPO_DIR` in the next cell to wherever Kaggle mounts it (typically `/kaggle/input/<dataset-name>`), then skip the clone cell.
4. Optional: add a second secret named `HF_TOKEN` (a HuggingFace [access token](https://huggingface.co/settings/tokens), read-only is enough), also toggled ON for this notebook, for higher Hub rate limits/faster downloads. Not required for any model used here, but the notebook will use it if present.
5. Kaggle's free tier: about 30 GPU-hours/week, 9-12h max per session. Gemma 4 **E4B** may not fit the T4 at all for *training* (its LoRA footprint has been reported elsewhere as ~17GB, over the T4's 16GB) -- the feasibility check cell below confirms this on the real hardware; E4B *baseline* eval (no LoRA) is much lighter and runs regardless. The **consistency** GRPO condition costs roughly double per step -- each full-training cell below sizes its own step count from a preceding timing-test cell's measured cost against a fixed time budget, specifically so this doesn't need a human to read a timing printout and hand-adjust the next cell.

This notebook is a thin runner around the actual project code (`eval.py`, `train_sft.py`, `train_grpo.py`) -- it doesn't reimplement any of the logic, so if you change the training/eval logic, edit those files and re-clone/re-sync rather than editing this notebook's cells directly.

In [1]:
import os, subprocess, sys

REPO_URL = "github.com/Vedang-P/neuro-symbolic-pathfinding.git"
REPO_DIR = "/kaggle/working/neuro-symbolic-pathfinding"

# GIT_TERMINAL_PROMPT=0: if auth still fails for some other reason (token
# lacks access to this repo, expired, etc.), git fails immediately with a
# clear error instead of hanging on an interactive password prompt that has
# nowhere to go in a notebook environment.
env = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print("Repo already present, pulling latest...")
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True, env=env)
elif not os.path.isdir(REPO_DIR):
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
        # Token must go in the PASSWORD slot (after the colon), not as a bare
        # username before @ -- a bare-username URL leaves git still needing a
        # password, which it then can't prompt for here ("could not read
        # Password ... No such device or address"). "x-access-token" as the
        # username is GitHub's documented convention for token-based HTTPS auth.
        clone_url = f"https://x-access-token:{token}@{REPO_URL}"
        subprocess.run(["git", "clone", clone_url, REPO_DIR], check=True, env=env)
        print("Cloned via GITHUB_TOKEN secret.")
    except Exception as e:
        print(f"Could not clone via secret ({type(e).__name__}: {e}).")
        print("Check: the secret is attached to this notebook (Add-ons -> Secrets -> make sure")
        print("it's toggled ON for this notebook specifically), the token hasn't expired, and it")
        print("has read access to this exact repo (fine-grained tokens need per-repo access grants).")
        print("If you attached the repo as a Kaggle Dataset instead, set REPO_DIR above to its mount")
        print("path (typically /kaggle/input/<dataset-name>) and re-run this cell.")
        raise

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print("Working dir:", os.getcwd())


Cloning into '/kaggle/working/neuro-symbolic-pathfinding'...


Cloned via GITHUB_TOKEN secret.
Working dir: /kaggle/working/neuro-symbolic-pathfinding


In [2]:
# Pulls in alphamaze_reference/ (github.com/menloresearch/visual-thinker) -- eval.py uses their
# real MazeBench scoring code directly from this submodule when present, falling back to a less
# faithful exact-match approach (with a loud warning) if it isn't.
subprocess.run(["git", "submodule", "update", "--init"], check=True)


Submodule 'alphamaze_reference' (https://github.com/menloresearch/visual-thinker.git) registered for path 'alphamaze_reference'
Cloning into '/kaggle/working/neuro-symbolic-pathfinding/alphamaze_reference'...


Submodule path 'alphamaze_reference': checked out 'bb74ee6fbf72b34321edcf2bb958921f694ab622'


CompletedProcess(args=['git', 'submodule', 'update', '--init'], returncode=0)

In [3]:
# Optional: read from Kaggle Secrets, never pasted here -- higher HF Hub rate
# limits/download speed, and required if any candidate model is gated.
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = hf_token
    os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token  # some older library versions check this name
    print("HF_TOKEN set from Kaggle secret.")
except Exception as e:
    print(f"No HF_TOKEN secret found ({type(e).__name__}) -- proceeding unauthenticated "
          "(fine for these public models, just slower/rate-limited).")


HF_TOKEN set from Kaggle secret.


In [4]:
# requirements.txt pins peft>=0.19.0, which ships its own Gemma4ClippableLinear
# support -- Gemma 4 no longer needs Unsloth at all (see hf_models.py's
# load_trainable_model() docstring), so there's no separate unsloth install here.
%pip install -q -r requirements.txt
# Kaggle's preinstalled wandb has been observed with its SDK and its own
# bundled generated protobuf file out of sync ("cannot import name 'Imports'
# from wandb.proto.wandb_telemetry_pb2"), which crashes on import -- trl's
# base_trainer.py checks wandb availability at import time, unconditionally,
# regardless of Unsloth. We never use wandb (report_to=[] everywhere) --
# removing it outright is the surest fix, on top of WANDB_DISABLED below.
%pip uninstall -q -y wandb


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 93.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 863.2/863.2 kB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 34.7 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [5]:
import os
# Belt and suspenders alongside hf_models.configure_quiet_logging() (which
# also sets these for every subprocess this notebook launches) -- set here
# too in case anything in the notebook's own kernel process ever imports
# wandb directly.
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")


CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


## Run infrastructure: `run_stage()`

Every pipeline stage below goes through this. It never raises -- a stage that OOMs, hits a bug, or has a transient HF Hub hiccup is logged and skipped over, not fatal to the rest of an unattended run. `required_paths` lets a stage declare "don't even try me if this input is missing" (e.g. a GRPO stage needing the preceding SFT stage's checkpoint), so a failure shows up as a clear, attributable skip reason in the log instead of a cryptic error three layers deep in someone else's library. `steps_for_budget()` reads a prior GRPO timing-test's measured per-step cost and sizes the following full run to fit a wall-clock budget automatically, which is what makes "no human between cells" actually work -- without it, the standard advice ("check the timing printout, adjust --max_steps") requires exactly the babysitting this notebook is built to avoid.

In [6]:
import json
import subprocess
import sys
import time
from pathlib import Path

RESULTS_DIR = Path("./results")
STAGE_LOG_PATH = RESULTS_DIR / "stage_log.json"
stage_results = json.loads(STAGE_LOG_PATH.read_text()) if STAGE_LOG_PATH.exists() else []

# Safety-ceiling timeouts (a hard kill-switch if something hangs) -- these are
# NOT the target duration for a stage, just an upper bound. Full GRPO runs
# size their own --max_steps well below their timeout via steps_for_budget().
TIMEOUT_QUICK = 30 * 60        # feasibility check, GRPO timing tests
# Measured directly on real hardware: MazeBench at 8192 max_new_tokens ran
# ~180s/maze on a T4 -- n=100 there alone is ~5h, and this pipeline has 9
# separate MazeBench calls, so eval.py's MazeBench floor came back down to
# 4096 tokens (already confirmed sufficient for a real 100/100-mazes-closed-
# their-think-tags run, see eval.py's --max_new_tokens help) and every
# MazeBench call below asks for n=20, not 100 -- still a real sample, just
# one that fits a Kaggle session across all 9 calls. GridRoute's own n=50
# is unchanged (shorter expected outputs, no evidence it's similarly slow).
TIMEOUT_EVAL = 60 * 60         # baseline/checkpoint evals
TIMEOUT_SFT = 90 * 60          # SFT warm-start
TIMEOUT_GRPO_FULL = 2 * 60 * 60  # full GRPO training runs, ceiling above their own time budget

# Target wall-clock budget per full GRPO training run -- lower this if your
# weekly quota is tight, raise it if you have room. Applied per condition, so
# the pricier "consistency" condition naturally gets fewer steps than
# "single"/"mixed" for the same budget, without any special-casing.
GRPO_BUDGET_MINUTES = 60


def _save_stage_log():
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    STAGE_LOG_PATH.write_text(json.dumps(stage_results, indent=2))


def run_stage(name, args, required_paths=None, timeout=None):
    """Run `python <args>` as a subprocess. Always returns, never raises."""
    print(f"\n{'='*70}\n\u25b6 {name}\n{'='*70}")
    for p in (required_paths or []):
        if not Path(p).exists():
            print(f"\u23ed  SKIPPED -- required input not found: {p}")
            stage_results.append({"stage": name, "status": "skipped", "reason": f"missing {p}"})
            _save_stage_log()
            return stage_results[-1]

    t0 = time.time()
    try:
        proc = subprocess.run([sys.executable] + args, timeout=timeout)
        elapsed_min = round((time.time() - t0) / 60, 1)
        if proc.returncode == 0:
            print(f"\n\u2705 DONE ({elapsed_min} min)")
            result = {"stage": name, "status": "ok", "elapsed_min": elapsed_min}
        else:
            print(f"\n\u274c FAILED (exit code {proc.returncode}, {elapsed_min} min) -- see output above")
            result = {"stage": name, "status": "failed", "exit_code": proc.returncode, "elapsed_min": elapsed_min}
    except subprocess.TimeoutExpired:
        elapsed_min = round((time.time() - t0) / 60, 1)
        print(f"\n\u23f1  TIMED OUT after {elapsed_min} min (limit {timeout/60:.0f} min)")
        result = {"stage": name, "status": "timeout", "elapsed_min": elapsed_min}
    except Exception as e:
        elapsed_min = round((time.time() - t0) / 60, 1)
        print(f"\n\u274c FAILED (exception: {e})")
        result = {"stage": name, "status": "error", "error": str(e), "elapsed_min": elapsed_min}

    stage_results.append(result)
    _save_stage_log()
    return result


def steps_for_budget(timing_dir, budget_minutes=GRPO_BUDGET_MINUTES, min_steps=20, default_steps=200):
    """Read a prior train_grpo.py run's timing.json and compute a step count
    that fits budget_minutes at that measured per-step cost. Falls back to
    default_steps if timing.json is missing (e.g. the timing test itself
    failed) -- better to attempt a bounded full run than skip it outright."""
    timing_path = Path(timing_dir) / "timing.json"
    if not timing_path.exists():
        print(f"  (no timing.json at {timing_dir}, using default {default_steps} steps)")
        return default_steps
    t = json.loads(timing_path.read_text())
    per_step = t.get("per_step_s", 0)
    if per_step <= 0:
        return default_steps
    steps = max(min_steps, int(budget_minutes * 60 / per_step))
    print(f"  (timing.json: {per_step:.1f}s/step -> {steps} steps fits a {budget_minutes}min budget)")
    return steps


## Feasibility check: does Gemma 4 E2B/E4B LoRA actually fit this GPU?

Don't assume the ~8-10GB (E2B) / ~17GB (E4B) numbers from elsewhere -- confirm on this exact hardware.

In [7]:
run_stage("Feasibility check (all candidate models)", ["check_finetune_feasibility.py"], timeout=TIMEOUT_QUICK)



▶ Feasibility check (all candidate models)
GPU: Tesla T4
Total VRAM: 15.64 GB
CUDA allocated before tests: 0.00 GB

Testing: deepseek-r1-distill-qwen-1.5b  (deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B)
  Loading + attaching LoRA (r=16)...


Loading weights: 100%|██████████| 339/339 [00:01<00:00, 217.50it/s]


  Backend: bnb_peft
  Running forward+backward (seq_len<=256, batch=1)...

  VRAM breakdown (summed across 2 GPU(s)):
    Before:       0.00 GB
    After LoRA:   2.62 GB  (+2.62)
    After fwd+bwd:2.72 GB
    Peak:         3.01 GB
    Est. GRPO:    4.21 GB  → ✅ FEASIBLE  (rough estimate, see docstring)

Testing: smollm2-1.7b  (HuggingFaceTB/SmolLM2-1.7B-Instruct)
  Loading + attaching LoRA (r=16)...


Loading weights: 100%|██████████| 218/218 [00:01<00:00, 178.72it/s]


  Backend: bnb_peft
  Running forward+backward (seq_len<=256, batch=1)...

  VRAM breakdown (summed across 2 GPU(s)):
    Before:       0.70 GB
    After LoRA:   2.00 GB  (+1.31)
    After fwd+bwd:2.08 GB
    Peak:         2.13 GB
    Est. GRPO:    2.98 GB  → ✅ FEASIBLE  (rough estimate, see docstring)

Testing: gemma4-e2b  (unsloth/gemma-4-E2B-it)
  Loading + attaching LoRA (r=16)...


Loading weights: 100%|██████████| 1951/1951 [00:03<00:00, 533.24it/s]


  Backend: bnb_peft
  Running forward+backward (seq_len<=256, batch=1)...

  VRAM breakdown (summed across 2 GPU(s)):
    Before:       0.85 GB
    After LoRA:   6.77 GB  (+5.92)
    After fwd+bwd:6.79 GB
    Peak:         6.86 GB
    Est. GRPO:    9.60 GB  → ✅ FEASIBLE  (rough estimate, see docstring)

Testing: gemma4-e4b  (unsloth/gemma-4-E4B-it)
  Loading + attaching LoRA (r=16)...


Loading weights: 100%|██████████| 2076/2076 [00:41<00:00, 50.22it/s] 


  Backend: bnb_peft
  Running forward+backward (seq_len<=256, batch=1)...

  VRAM breakdown (summed across 2 GPU(s)):
    Before:       0.99 GB
    After LoRA:   9.34 GB  (+8.35)
    After fwd+bwd:9.37 GB
    Peak:         9.43 GB
    Est. GRPO:    13.20 GB  → ✅ FEASIBLE  (rough estimate, see docstring)

MODEL                    BACKEND     LOAD+LORA   PEAK   GRPO  VERDICT
--------------------------------------------------------------------------------
deepseek-r1-distill-qwen-1.5b bnb_peft         2.6G   3.0G   4.2G  ✅ FEASIBLE
smollm2-1.7b             bnb_peft         1.3G   2.1G   3.0G  ✅ FEASIBLE
gemma4-e2b               bnb_peft         5.9G   6.9G   9.6G  ✅ FEASIBLE
gemma4-e4b               bnb_peft         8.3G   9.4G  13.2G  ✅ FEASIBLE

✅ DONE (4.0 min)


{'stage': 'Feasibility check (all candidate models)',
 'status': 'ok',
 'elapsed_min': 4.0}

## Phase 1: baselines -- Gemma 4 E2B/E4B + AlphaMaze on MazeBench and GridRoute 5x5

AlphaMaze should land near its published 93% on MazeBench (using their real scoring code via the submodule) -- this is the harness sanity check. Gemma 4's numbers on both benchmarks are the actual open question: no SLM has been tested on GridRoute anywhere in the literature found so far.

In [8]:
ALPHAMAZE_LOCAL_PATH = "data/models/alphamaze-v0.2-1.5b"
os.makedirs("data/models", exist_ok=True)
try:
    if not os.path.isdir(ALPHAMAZE_LOCAL_PATH):
        from huggingface_hub import snapshot_download
        snapshot_download("homebrewltd/AlphaMaze-v0.2-1.5B", local_dir=ALPHAMAZE_LOCAL_PATH)
    print("AlphaMaze checkpoint ready.")
except Exception as e:
    print(f"\u26a0\ufe0f  AlphaMaze checkpoint download failed ({e}) -- AlphaMaze-specific "
          "stages below will be skipped (they require-path-check this directory).")


Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

AlphaMaze checkpoint ready.


This specific MazeBench call is the Phase 1 replication check -- its entire job is confirming this harness reproduces AlphaMaze's published ~93%, so it deliberately doesn't use the faster settings below: full precision (`--no_4bit`, removes quantization as a confound entirely) and a generous token budget (`--max_new_tokens 16384` -- their real reasoning style re-renders the whole board state after every single step, which is genuinely token-expensive; better to run long than risk truncating a maze that was actually on track and scoring it as a failure it didn't earn). Slower, on purpose -- this one cell prioritizes matching their number over speed. Everything after this cell goes back to the faster settings, since there's no published number to match for Gemma 4 on either benchmark.

In [9]:
run_stage("AlphaMaze baseline: MazeBench (full precision, faithful replication)",
          ["eval.py", "--model", "alphamaze", "--benchmark", "mazebench", "--n", "30",
           "--no_4bit", "--max_new_tokens", "16384", "--output_dir", "./results/eval"],
          required_paths=[ALPHAMAZE_LOCAL_PATH], timeout=3 * 60 * 60)
run_stage("AlphaMaze baseline: GridRoute NL",
          ["eval.py", "--model", "alphamaze", "--benchmark", "gridroute-nl", "--grid_size", "5",
           "--n", "50", "--output_dir", "./results/eval"],
          required_paths=[ALPHAMAZE_LOCAL_PATH], timeout=TIMEOUT_EVAL)



▶ AlphaMaze baseline: MazeBench (full precision, faithful replication)
Model: ./data/models/alphamaze-v0.2-1.5b
4bit=False  backend=hf  benchmark=mazebench


MazeBench:   0%|          | 0/30 [00:00<?, ?maze/s]

  [1/30] generating (streamed below)...
<think>
Step 1: Go up
<|0-0|><|up_left_wall|><|blank|><|0-1|><|up_wall|><|blank|><|0-2|><|up_right_wall|><|blank|><|0-3|><|up_down_left_wall|><|blank|><|0-4|><|up_right_wall|><|blank|>
<|1-0|><|down_left_right_wall|><|blank|><|1-1|><|left_right_wall|><|up|><|1-2|><|down_left_wall|><|blank|><|1-3|><|up_right_wall|><|blank|><|1-4|><|left_right_wall|><|blank|>
<|2-0|><|up_left_wall|><|blank|><|2-1|><|down_right_wall|><|blank|><|2-2|><|up_left_wall|><|blank|><|2-3|><|down_right_wall|><|blank|><|2-4|><|left_right_wall|><|blank|>
<|3-0|><|down_left_right_wall|><|target|><|3-1|><|up_left_right_wall|><|blank|><|3-2|><|down_left_wall|><|blank|><|3-3|><|up_right_wall|><|blank|><|3-4|><|left_right_wall|><|blank|>
<|4-0|><|up_down_left_wall|><|blank|><|4-1|><|down_wall|><|blank|><|4-2|><|up_down_wall|><|blank|><|4-3|><|down_wall|><|blank|><|4-4|><|down_right_wall|><|blank|>

Step 2: Go right
<|0-0|><|up_left_wall|><|blank|><|0-1|><|up_wall|><|right|><|0-2|><

MazeBench:   3%|▎         | 1/30 [00:44<21:43, 44.97s/maze, acc=0/1]


  [1/30] x   level=easy pred_len=12 gt_len=3
  [2/30] generating (streamed below)...
<think>
Step 1: Go down
<|0-0|><|up_left_wall|><|blank|><|0-1|><|up_down_wall|><|blank|><|0-2|><|up_down_wall|><|blank|><|0-3|><|up_wall|><|blank|><|0-4|><|up_right_wall|><|target|>
<|1-0|><|down_left_wall|><|blank|><|1-1|><|up_right_wall|><|blank|><|1-2|><|up_left_wall|><|blank|><|1-3|><|down_right_wall|><|blank|><|1-4|><|left_right_wall|><|down|>
<|2-0|><|up_down_left_wall|><|blank|><|2-1|><|down_right_wall|><|blank|><|2-2|><|left_right_wall|><|blank|><|2-3|><|up_left_right_wall|><|blank|><|2-4|><|left_right_wall|><|blank|>
<|3-0|><|up_left_right_wall|><|blank|><|3-1|><|up_left_wall|><|blank|><|3-2|><|down_right_wall|><|blank|><|3-3|><|left_right_wall|><|blank|><|3-4|><|left_right_wall|><|blank|>
<|4-0|><|down_left_wall|><|blank|><|4-1|><|down_right_wall|><|blank|><|4-2|><|up_down_left_wall|><|blank|><|4-3|><|down_wall|><|blank|><|4-4|><|down_right_wall|><|blank|>

Step 2: Go down
<|0-0|><|up_left_w

...

GridRoute-nl-5x5:  98%|█████████▊| 49/50 [20:58<00:25, 25.64s/task, optimal=7/49, valid=8/49]

<think>
First, I need to navigate from the starting position (0,2) to the target (2,4) on a 5x5 grid. The only obstacle is at (0,3), and I can only move up, down, left, or right.

I'll start by moving right to clear the path. This takes me to (0,3), which is blocked, so I'll move left instead. Now, I can move up, down, or right. Moving up takes me to (0,4), which is a clear path. From (0,4), I can move right to (0,5), but that's the edge of the grid. Instead, I'll move right to (0,1), which is a better position.

From (0,1), I can move up to (1,1) or right to (0,2). Moving up to (1,1) seems more efficient. From (1,1), I can move up to (2,1) or right to (1,2). Moving up to (2,1) brings me closer to the target. From (2,1), I can move up to (3,1) or right to (2,2). Moving up to (3,1) is a step in the wrong direction, so I'll move right to (2,2).

From (2,2), I can move up to (1,2) or right to (2,3). Moving up to (1,2) is a better option. From (1,2), I can move up to (0,2) or right to (1,3

GridRoute-nl-5x5: 100%|██████████| 50/50 [23:45<00:00, 28.52s/task, optimal=7/50, valid=8/50]



✅ DONE (24.0 min)


{'stage': 'AlphaMaze baseline: GridRoute NL',
 'status': 'ok',
 'elapsed_min': 24.0}

In [10]:
run_stage("Gemma 4 E2B baseline: MazeBench",
          ["eval.py", "--model", "gemma4-e2b", "--benchmark", "mazebench", "--n", "20",
           "--output_dir", "./results/eval"], timeout=TIMEOUT_EVAL)
run_stage("Gemma 4 E2B baseline: GridRoute NL",
          ["eval.py", "--model", "gemma4-e2b", "--benchmark", "gridroute-nl", "--grid_size", "5",
           "--n", "50", "--output_dir", "./results/eval"], timeout=TIMEOUT_EVAL)



▶ Gemma 4 E2B baseline: MazeBench
Model: unsloth/gemma-4-E2B-it
4bit=True  backend=hf  benchmark=mazebench


MazeBench:   0%|          | 0/20 [00:00<?, ?maze/s]

  [1/20] generating (streamed below)...
thought
The user wants me to solve a maze.
I need to determine the sequence of movements (up, down, left, right) from the 'origin' to the 'target'.

First, I need to parse the maze representation and determine the coordinates, walls, origin, and target.

The maze is represented as a sequence of tokens. I will use (row, col) coordinates, starting from (0, 0).

Maze structure:
Row 0: <|0-0|><|up_left_wall|><|blank|><|0-1|><|up_wall|><|blank|><|0-2|><|up_right_wall|><|blank|><|0-3|><|up_down_left_wall|><|blank|><|0-4|><|up_right_wall|><|blank|>
Row 1: <|1-0|><|down_left_right_wall|><|blank|><|1-1|><|left_right_wall|><|origin|><|1-2|><|down_left_wall|><|blank|><|1-3|><|up_right_wall|><|blank|><|1-4|><|left_right_wall|><|blank|>
Row 2: <|2-0|><|up_left_wall|><|blank|><|2-1|><|down_right_wall|><|blank|><|2-2|><|up_left_wall|><|blank|><|2-3|><|down_right_wall|><|blank|><|2-4|><|left_right_wall|><|blank|>
Row 3: <|3-0|><|down_left_right_wall|><|target|><

MazeBench:   5%|▌         | 1/20 [08:28<2:40:54, 508.15s/maze, acc=0/1]


  [1/20] no-answer level=easy pred_len=0 gt_len=3
  [2/20] generating (streamed below)...
thought
The user wants me to solve a maze.
I need to represent the maze, track the current position, and determine the sequence of moves from 'origin' to 'target'.

Maze representation:
The maze is given as a sequence of tokens. I need to parse this into a grid structure and identify walls, empty spaces, origin, and target.

Let's define coordinates (row, col).

Maze structure parsing:

Row 0: <|0-0|><|up_left_wall|><|blank|><|0-1|><|up_down_wall|><|blank|><|0-2|><|up_down_wall|><|blank|><|0-3|><|up_wall|><|blank|><|0-4|><|up_right_wall|><|target|>
Row 1: <|1-0|><|down_left_wall|><|blank|><|1-1|><|up_right_wall|><|blank|><|1-2|><|up_left_wall|><|blank|><|1-3|><|down_right_wall|><|blank|><|1-4|><|left_right_wall|><|origin|>
Row 2: <|2-0|><|up_down_left_wall|><|blank|><|2-1|><|down_right_wall|><|blank|><|2-2|><|left_right_wall|><|blank|><|2-3|><|up_left_right_wall|><|blank|><|2-4|><|left_right_wall

...

GridRoute-nl-5x5:  42%|████▏     | 21/50 [55:45<1:15:47, 156.83s/task, optimal=21/21, valid=21/21]


  [21/50] OPT   start=(0, 1) goal=(4, 1) opt_len=4
  [22/50] generating (streamed below)...
thought
The user wants me to find a path from the starting point (1,4) to the target (0,0) on a 5x5 grid.
The grid coordinates are (row, column), where rows are indexed 0 to 4 and columns 0 to 4.
The grid size is 5x5.
The obstacle is at (1,0).
Allowed moves are up, down, left, right.

Start: S = (1, 4)
Target: T = (0, 0)
Obstacle: O = (1, 0)

I need to find a sequence of valid moves. Since the grid is small and the goal is to find *a* path, I will use a search algorithm (like BFS or simply pathfinding by inspection) while avoiding the obstacle.

Grid representation (R, C):
(0,0) (0,1) (0,2) (0,3) (0,4)
(1,0) X (1,1) (1,2) (1,3) (1,4)
(2,0) (2,1) (2,2) (2,3) (2,4)
(3,0) (3,1) (3,2) (3,3) (3,4)
(4,0) (4,1) (4,2) (4,3) (4,4)

Obstacle at (1, 0).

Let's try to move towards (0,0).

Start (1, 4).
Possible moves:
1. Up: (0, 4)
2. Down: (2, 4)
3. Left: (1, 3)
4. Right: (1, 5) -> Invalid (out of bounds,

GridRoute-nl-5x5:  44%|████▍     | 22/50 [58:15<1:12:13, 154.78s/task, optimal=22/22, valid=22/22]


  [22/50] OPT   start=(1, 4) goal=(0, 0) opt_len=5
  [23/50] generating (streamed below)...
thought
The user wants me to find a path from the starting point (2,0) to the target point (4,2) on a 5x5 grid.
The grid coordinates are (row, column), typically indexed from (0,0) to (4,4).
The obstacles are at (2,3).
Allowed moves are up, down, left, right (Manhattan distance movement).

Start: S = (2, 0)
Target: T = (4, 2)
Obstacle: O = (2, 3)

Grid size: 5x5.

I will use Breadth-First Search (BFS) or simply try to find a path, prioritizing shortest paths if possible, but any valid path is acceptable.

Let's represent the grid and track visited cells.

Grid visualization (R, C):
(0,0) (0,1) (0,2) (0,3) (0,4)
(1,0) (1,1) (1,2) (1,3) (1,4)
(2,0) (2,1) (2,2) (2,3)X (2,4)
(3,0) (3,1) (3,2) (3,3) (3,4)
(4,0) (4,1) (4,2) (4,3) (4,4)

Start (2, 0). Target (4, 2). Obstacle (2, 3).

Step 1: From (2, 0).
Possible moves: (1, 0), (3, 0), (2, 1).

Path 1: (2, 0) -> (2, 1)
From (2, 1). Possible moves: (1,

{'stage': 'Gemma 4 E2B baseline: GridRoute NL',
 'status': 'timeout',
 'elapsed_min': 60.0}

E4B baseline (inference-only, no LoRA) is much lighter than E4B *training* -- run this regardless of what the feasibility check said about training, since it doesn't need LoRA attachment at all, just enough VRAM to hold the weights at 4-bit.

In [11]:
run_stage("Gemma 4 E4B baseline: MazeBench",
          ["eval.py", "--model", "gemma4-e4b", "--benchmark", "mazebench", "--n", "20",
           "--output_dir", "./results/eval"], timeout=TIMEOUT_EVAL)
run_stage("Gemma 4 E4B baseline: GridRoute NL",
          ["eval.py", "--model", "gemma4-e4b", "--benchmark", "gridroute-nl", "--grid_size", "5",
           "--n", "50", "--output_dir", "./results/eval"], timeout=TIMEOUT_EVAL)



▶ Gemma 4 E4B baseline: MazeBench
Model: unsloth/gemma-4-E4B-it
4bit=True  backend=hf  benchmark=mazebench


MazeBench:   0%|          | 0/20 [00:00<?, ?maze/s]

  [1/20] generating (streamed below)...
thought
The user wants me to solve a maze represented by a series of tokens.
I need to find the sequence of movements (<|up|>, <|down|>, <|left|>, <|right|>) from the origin to the target.
I must output only the move tokens, separated by spaces.

The maze structure is a grid of cells, and each cell contains information about its coordinates, walls, and special markers (origin/target).

Let's analyze the maze structure. It seems to be a 5x5 grid of cells, where each cell description is followed by a separator (implied by the structure, likely a cell boundary).

The input format is a continuous string of tokens. I need to parse this into a grid representation.

The maze is given as:
Row 0: <|0-0|><|up_left_wall|><|blank|><|0-1|><|up_wall|><|blank|><|0-2|><|up_right_wall|><|blank|><|0-3|><|up_down_left_wall|><|blank|><|0-4|><|up_right_wall|><|blank|>
Row 1: <|1-0|><|down_left_right_wall|><|blank|><|1-1|><|left_right_wall|><|origin|><|1-2|><|down_lef

MazeBench:   5%|▌         | 1/20 [10:50<3:25:55, 650.31s/maze, acc=0/1]


  [1/20] no-answer level=easy pred_len=0 gt_len=3
  [2/20] generating (streamed below)...
thought
The user wants me to solve a maze represented by a series of tokens.
I need to find the sequence of movements (<|up|>, <|down|>, <|left|>, <|right|>) from the origin to the target.
I must output only the movement tokens, separated by spaces.

The maze is a 5x5 grid (5 rows, 5 columns).

Let's map the grid structure.
The input is a sequence of tokens representing cells. Each row of the input corresponds to a row in the grid.

Grid dimensions: 5 rows (0 to 4), 5 columns (0 to 4).

Row 0:
<|0-0|> <|up_left_wall|> <|blank|> <|0-1|> <|up_down_wall|> <|blank|> <|0-2|> <|up_down_wall|> <|blank|> <|0-3|> <|up_wall|> <|blank|> <|0-4|> <|up_right_wall|> <|target|>

Wait, the input structure seems to be:
Cell (R, C) -> Cell (R, C+1) -> ...

Let's re-examine the input structure. It looks like a flattened representation where each cell is defined, and the walls define connectivity.

The input is:
Row 

...

GridRoute-nl-5x5:  22%|██▏       | 11/50 [52:31<3:36:28, 333.03s/task, optimal=11/11, valid=11/11]


  [11/50] OPT   start=(0, 4) goal=(4, 1) opt_len=7
  [12/50] generating (streamed below)...
thought
The user wants me to find a path from the starting point (2, 4) to the goal point (4, 1) on a 5x5 grid.
The grid coordinates are likely (row, column), where rows and columns range from 0 to 4.
The grid size is 5x5.
Start: (2, 4)
Goal: (4, 1)
Obstacle: (1, 2)

The allowed moves are up, down, left, right.

Let's define the grid boundaries: Rows {0, 1, 2, 3, 4}, Columns {0, 1, 2, 3, 4}.

Start $S = (2, 4)$
Goal $G = (4, 1)$
Obstacle $O = (1, 2)$

I will use Breadth-First Search (BFS) or a similar pathfinding approach to find the shortest path, although any valid path is acceptable. Since the grid is small, a manual search or BFS simulation should be efficient.

**Step 1: Initialization**
Queue: [( (2, 4), [(2, 4)] )]  # (current_position, path_so_far)
Visited: {(2, 4)}

**Step 2: BFS Execution**

1. **Dequeue:** ((2, 4), [(2, 4)])
   Neighbors of (2, 4):
   (1, 4) - Valid
   (3, 4) - Valid

GridRoute-nl-5x5:  24%|██▍       | 12/50 [56:40<3:14:53, 307.71s/task, optimal=12/12, valid=12/12]


  [12/50] OPT   start=(2, 4) goal=(4, 1) opt_len=5
  [13/50] generating (streamed below)...
thought
The user wants me to find a path from the starting point (4, 0) to the goal point (0, 2) on a 5x5 grid.
The grid coordinates are likely (row, column), where rows range from 0 to 4 and columns range from 0 to 4.
The grid size is 5x5.
Start: (4, 0)
Goal: (0, 2)
Obstacle: (1, 2)

I need to use Breadth-First Search (BFS) or a similar pathfinding algorithm (like A*) to find the shortest path, avoiding the obstacle. Since the grid is small, BFS should be sufficient.

Grid representation:
Rows (r): 0 to 4
Cols (c): 0 to 4

Start S = (4, 0)
Goal G = (0, 2)
Obstacle O = (1, 2)

Let's define the possible moves: Up (r-1), Down (r+1), Left (c-1), Right (c+1).

**BFS Implementation:**

1. Initialize Queue: Queue = [(start_node, path_to_node)]
2. Initialize Visited Set: Visited = {start_node}

**Step 1: Initialization**
Queue = [((4, 0), [(4, 0)])]
Visited = {(4, 0)}

**Step 2: BFS Loop**

**Iteratio

{'stage': 'Gemma 4 E4B baseline: GridRoute NL',
 'status': 'timeout',
 'elapsed_min': 60.0}

## Phase 1b: full-precision ablation for Gemma 4 (how much does 4-bit cost?)

Gemma 4's baselines above ran 4-bit-quantized -- necessary for *training* (LoRA/GRPO need the VRAM headroom), but baseline inference has no such requirement, so it's worth checking directly rather than assuming the gap is small. Same `--n`/seed as the corresponding 4-bit cell above, so this is a paired comparison on identical tasks, not just a same-average different-sample one. `eval.py` records `load_in_4bit` in its output JSON specifically so the aggregation table below can show both side by side. Gemma 4 E4B at full precision is the one most likely to not fit the T4 at all (4-bit exists exactly to avoid this) -- if that cell fails, that itself is a useful, reportable data point, not a notebook bug.

AlphaMaze isn't part of this ablation at all: it always runs full precision, enforced directly in `eval.py` regardless of any `--load_in_4bit` flag, not just by convention in this notebook. It's the one fixed reference point every other number here gets compared against and it's never trained further in Phase 2, so there's no training-time VRAM constraint forcing a quantization tradeoff the way there is for Gemma 4 -- no reason to ever accept the noise 4-bit would add to its numbers.

In [12]:
run_stage("[full-precision] Gemma 4 E2B baseline: MazeBench",
          ["eval.py", "--model", "gemma4-e2b", "--benchmark", "mazebench", "--n", "20",
           "--no_4bit", "--output_dir", "./results/eval"], timeout=TIMEOUT_EVAL)
run_stage("[full-precision] Gemma 4 E2B baseline: GridRoute NL",
          ["eval.py", "--model", "gemma4-e2b", "--benchmark", "gridroute-nl", "--grid_size", "5",
           "--n", "50", "--no_4bit", "--output_dir", "./results/eval"], timeout=TIMEOUT_EVAL)



▶ [full-precision] Gemma 4 E2B baseline: MazeBench
Model: unsloth/gemma-4-E2B-it
4bit=False  backend=hf  benchmark=mazebench


MazeBench:   0%|          | 0/20 [00:00<?, ?maze/s]

  [1/20] generating (streamed below)...
thought
The user wants me to solve a maze and output the sequence of movements from the origin to the target.

First, I need to parse the maze representation. The maze is given as a series of tokens.

Maze structure:
Row 0: <|0-0|> <|up_left_wall|> <|blank|> <|0-1|> <|up_wall|> <|blank|> <|0-2|> <|up_right_wall|> <|blank|> <|0-3|> <|up_down_left_wall|> <|blank|> <|0-4|> <|up_right_wall|> <|blank|>
Row 1: <|1-0|> <|down_left_right_wall|> <|blank|> <|1-1|> <|left_right_wall|> <|origin|> <|1-2|> <|down_left_wall|> <|blank|> <|1-3|> <|up_right_wall|> <|blank|> <|1-4|> <|left_right_wall|> <|blank|>
Row 2: <|2-0|> <|up_left_wall|> <|blank|> <|2-1|> <|down_right_wall|> <|blank|> <|2-2|> <|up_left_wall|> <|blank|> <|2-3|> <|down_right_wall|> <|blank|> <|2-4|> <|left_right_wall|> <|blank|>
Row 3: <|3-0|> <|down_left_right_wall|> <|target|> <|3-1|> <|up_left_right_wall|> <|blank|> <|3-2|> <|down_left_wall|> <|blank|> <|3-3|> <|up_right_wall|> <|blank|> <|3

MazeBench:   5%|▌         | 1/20 [05:31<1:44:54, 331.30s/maze, acc=0/1]


  [1/20] no-answer level=easy pred_len=0 gt_len=3
  [2/20] generating (streamed below)...
thought
The user wants me to solve a maze and output the sequence of movements from the origin to the target.

First, I need to parse the maze representation. The maze is given as a series of tokens. I need to determine the starting position (origin) and the goal position (target).

Maze representation:
Row 0: <|0-0|> <|up_left_wall|> <|blank|> <|0-1|> <|up_down_wall|> <|blank|> <|0-2|> <|up_down_wall|> <|blank|> <|0-3|> <|up_wall|> <|blank|> <|0-4|> <|up_right_wall|> <|target|>
Row 1: <|1-0|> <|down_left_wall|> <|blank|> <|1-1|> <|up_right_wall|> <|blank|> <|1-2|> <|up_left_wall|> <|blank|> <|1-3|> <|down_right_wall|> <|blank|> <|1-4|> <|left_right_wall|> <|origin|>
Row 2: <|2-0|> <|up_down_left_wall|> <|blank|> <|2-1|> <|down_right_wall|> <|blank|> <|2-2|> <|left_right_wall|> <|blank|> <|2-3|> <|up_left_right_wall|> <|blank|> <|2-4|> <|left_right_wall|> <|blank|>
Row 3: <|3-0|> <|up_left_right_

...

GridRoute-nl-5x5:  58%|█████▊    | 29/50 [55:56<40:12, 114.87s/task, optimal=28/29, valid=28/29]


  [29/50] INVAL start=(0, 3) goal=(2, 2) opt_len=3
  [30/50] generating (streamed below)...
thought
The user wants me to find a path from a starting point $S=(2,0)$ to a target point $T=(4,3)$ on a $5 \times 5$ grid.
The grid coordinates are typically (row, column), where rows are indexed 0 to 4 and columns 0 to 4.

Grid size: 5x5.
Start $S = (2, 0)$.
Target $T = (4, 3)$.
Obstacle $O = \{(1, 1)\}$.

Allowed moves: Up, Down, Left, Right (4-connectivity).

I will use Breadth-First Search (BFS) or A* (since it's a simple grid, BFS is sufficient) to find the shortest path, although any valid path is acceptable.

Grid representation (R, C):
(0,0) (0,1) (0,2) (0,3) (0,4)
(1,0) (1,1) (1,2) (1,3) (1,4)
(2,0) (2,1) (2,2) (2,3) (2,4)
(3,0) (3,1) (3,2) (3,3) (3,4)
(4,0) (4,1) (4,2) (4,3) (4,4)

Start: (2, 0)
Target: (4, 3)
Obstacle: (1, 1)

Step 1: Initialize BFS. Queue = [(2, 0)]. Visited = {(2, 0)}.

Step 2: Explore from (2, 0).
Neighbors of (2, 0):
(1, 0) - Valid. Add to queue.
(3, 0) - Valid

GridRoute-nl-5x5:  60%|██████    | 30/50 [59:28<48:04, 144.23s/task, optimal=29/30, valid=29/30]


  [30/50] OPT   start=(2, 0) goal=(4, 3) opt_len=5
  [31/50] generating (streamed below)...
thought
The user wants me to find a path from a starting point $S=(2,0)$ to a target point $T=(4,3)$ on a 5x5 grid.
The grid coordinates are (row, column), typically indexed from (0,0) to (4,4).
The obstacles are at $O = \{(0,1)\}$.
Allowed moves are up, down, left, right (4-connectivity).

Grid size: 5x5. Rows 0-4, Columns 0-4.
Start $S = (2, 0)$.
Target $T = (4, 3)$.
Obstacle $O = \{(0, 1)\}$.

I will use Breadth-First Search (BFS) or simply try to 
⏱  TIMED OUT after 60.0 min (limit 60 min)


{'stage': '[full-precision] Gemma 4 E2B baseline: GridRoute NL',
 'status': 'timeout',
 'elapsed_min': 60.0}

In [13]:
run_stage("[full-precision] Gemma 4 E4B baseline: MazeBench",
          ["eval.py", "--model", "gemma4-e4b", "--benchmark", "mazebench", "--n", "20",
           "--no_4bit", "--output_dir", "./results/eval"], timeout=TIMEOUT_EVAL)
run_stage("[full-precision] Gemma 4 E4B baseline: GridRoute NL",
          ["eval.py", "--model", "gemma4-e4b", "--benchmark", "gridroute-nl", "--grid_size", "5",
           "--n", "50", "--no_4bit", "--output_dir", "./results/eval"], timeout=TIMEOUT_EVAL)



▶ [full-precision] Gemma 4 E4B baseline: MazeBench
Model: unsloth/gemma-4-E4B-it
4bit=False  backend=hf  benchmark=mazebench


Loading weights:  68%|██████▊   | 1416/2076 [00:17<00:08, 80.69it/s]
Traceback (most recent call last):
  File "/kaggle/working/neuro-symbolic-pathfinding/eval.py", line 329, in <module>
    main()
  File "/kaggle/working/neuro-symbolic-pathfinding/eval.py", line 298, in main
    adapter_path=args.checkpoint or None).load()
                                          ^^^^^^
  File "/kaggle/working/neuro-symbolic-pathfinding/hf_models.py", line 263, in load
    self.model = AutoModelForCausalLM.from_pretrained(
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/models/auto/auto_factory.py", line 402, in from_pretrained
    return model_class.from_pretrained(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py", line 4368, in from_pretrained
    loading_info, disk_offload_index = cls._load_pretrained_model(model, state_dict, checkpoint_files, load_config)
    


❌ FAILED (exit code 1, 0.6 min) -- see output above

▶ [full-precision] Gemma 4 E4B baseline: GridRoute NL
Model: unsloth/gemma-4-E4B-it
4bit=False  backend=hf  benchmark=gridroute-nl


Loading weights:  68%|██████▊   | 1416/2076 [00:04<00:02, 307.03it/s]
Traceback (most recent call last):
  File "/kaggle/working/neuro-symbolic-pathfinding/eval.py", line 329, in <module>
    main()
  File "/kaggle/working/neuro-symbolic-pathfinding/eval.py", line 298, in main
    adapter_path=args.checkpoint or None).load()
                                          ^^^^^^
  File "/kaggle/working/neuro-symbolic-pathfinding/hf_models.py", line 263, in load
    self.model = AutoModelForCausalLM.from_pretrained(
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/models/auto/auto_factory.py", line 402, in from_pretrained
    return model_class.from_pretrained(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py", line 4368, in from_pretrained
    loading_info, disk_offload_index = cls._load_pretrained_model(model, state_dict, checkpoint_files, load_config)
   


❌ FAILED (exit code 1, 0.4 min) -- see output above


{'stage': '[full-precision] Gemma 4 E4B baseline: GridRoute NL',
 'status': 'failed',
 'exit_code': 1,
 'elapsed_min': 0.4}

## Phase 2, recipe 1: single-format (GridRoute NL only) on Gemma 4 E2B

SFT warm-start, a short GRPO timing test, then a full run sized to fit `GRPO_BUDGET_MINUTES` at the timing test's measured per-step cost -- see the run-infrastructure cell above for why this is computed rather than manually tuned.

In [14]:
SFT_SINGLE_DIR = "./results/sft_gemma4-e2b_single"
run_stage("SFT: single-format (Gemma 4 E2B)",
          ["train_sft.py", "--model", "gemma4-e2b", "--format", "nl", "--grid_size", "5",
           "--n_tasks", "400", "--epochs", "1", "--output_dir", SFT_SINGLE_DIR], timeout=TIMEOUT_SFT)



▶ SFT: single-format (Gemma 4 E2B)
Model: unsloth/gemma-4-E2B-it
Format: nl  Tasks: 400  Grid: 5x5  Epochs: 1  LR: 1e-05


Loading weights: 100%|██████████| 1951/1951 [00:38<00:00, 50.07it/s] 


Model + LoRA ready (backend=bnb_peft)
Building SFT dataset (nl, 400 tasks, 5x5)...
  400 training rows.


Map: 100%|██████████| 400/400 [00:00<00:00, 12035.91 examples/s]
Traceback (most recent call last):
  File "/kaggle/working/neuro-symbolic-pathfinding/train_sft.py", line 181, in <module>
    main()
  File "/kaggle/working/neuro-symbolic-pathfinding/train_sft.py", line 149, in main
    training_args = SFTConfig(
                    ^^^^^^^^^^
TypeError: SFTConfig.__init__() got an unexpected keyword argument 'max_seq_length'



❌ FAILED (exit code 1, 1.0 min) -- see output above


{'stage': 'SFT: single-format (Gemma 4 E2B)',
 'status': 'failed',
 'exit_code': 1,
 'elapsed_min': 1.0}

In [15]:
GRPO_SINGLE_TIMING_DIR = "./results/grpo_gemma4-e2b_single_timing"
run_stage("GRPO timing test: single-format",
          ["train_grpo.py", "--model", "gemma4-e2b", "--adapter_path", SFT_SINGLE_DIR,
           "--condition", "single", "--grid_size", "5", "--n_tasks", "20", "--max_steps", "20",
           "--output_dir", GRPO_SINGLE_TIMING_DIR],
          required_paths=[SFT_SINGLE_DIR], timeout=TIMEOUT_QUICK)



▶ GRPO timing test: single-format
⏭  SKIPPED -- required input not found: ./results/sft_gemma4-e2b_single


{'stage': 'GRPO timing test: single-format',
 'status': 'skipped',
 'reason': 'missing ./results/sft_gemma4-e2b_single'}

In [16]:
GRPO_SINGLE_DIR = "./results/grpo_gemma4-e2b_single"
single_steps = steps_for_budget(GRPO_SINGLE_TIMING_DIR)
run_stage(f"GRPO: single-format full run ({single_steps} steps)",
          ["train_grpo.py", "--model", "gemma4-e2b", "--adapter_path", SFT_SINGLE_DIR,
           "--condition", "single", "--grid_size", "5", "--n_tasks", "50",
           "--max_steps", str(single_steps), "--output_dir", GRPO_SINGLE_DIR],
          required_paths=[SFT_SINGLE_DIR], timeout=TIMEOUT_GRPO_FULL)


  (no timing.json at ./results/grpo_gemma4-e2b_single_timing, using default 200 steps)

▶ GRPO: single-format full run (200 steps)
⏭  SKIPPED -- required input not found: ./results/sft_gemma4-e2b_single


{'stage': 'GRPO: single-format full run (200 steps)',
 'status': 'skipped',
 'reason': 'missing ./results/sft_gemma4-e2b_single'}

In [17]:
run_stage("Eval single-format checkpoint: MazeBench",
          ["eval.py", "--model", "gemma4-e2b", "--checkpoint", GRPO_SINGLE_DIR,
           "--benchmark", "mazebench", "--n", "20", "--output_dir", "./results/eval"],
          required_paths=[GRPO_SINGLE_DIR], timeout=TIMEOUT_EVAL)
run_stage("Eval single-format checkpoint: GridRoute NL",
          ["eval.py", "--model", "gemma4-e2b", "--checkpoint", GRPO_SINGLE_DIR,
           "--benchmark", "gridroute-nl", "--grid_size", "5", "--n", "50", "--output_dir", "./results/eval"],
          required_paths=[GRPO_SINGLE_DIR], timeout=TIMEOUT_EVAL)



▶ Eval single-format checkpoint: MazeBench
⏭  SKIPPED -- required input not found: ./results/grpo_gemma4-e2b_single

▶ Eval single-format checkpoint: GridRoute NL
⏭  SKIPPED -- required input not found: ./results/grpo_gemma4-e2b_single


{'stage': 'Eval single-format checkpoint: GridRoute NL',
 'status': 'skipped',
 'reason': 'missing ./results/grpo_gemma4-e2b_single'}

## Phase 2, recipe 2: mixed-format (NL + token, naive)

Does training on both formats (interleaved, same underlying grids) do better than single-format alone -- for either benchmark?

In [18]:
SFT_MIXED_DIR = "./results/sft_gemma4-e2b_mixed"
run_stage("SFT: mixed-format (Gemma 4 E2B)",
          ["train_sft.py", "--model", "gemma4-e2b", "--format", "mixed", "--grid_size", "5",
           "--n_tasks", "400", "--epochs", "1", "--output_dir", SFT_MIXED_DIR], timeout=TIMEOUT_SFT)



▶ SFT: mixed-format (Gemma 4 E2B)
Model: unsloth/gemma-4-E2B-it
Format: mixed  Tasks: 400  Grid: 5x5  Epochs: 1  LR: 1e-05


Loading weights: 100%|██████████| 1951/1951 [00:04<00:00, 426.36it/s] 


Model + LoRA ready (backend=bnb_peft)
Building SFT dataset (mixed, 400 tasks, 5x5)...
  800 training rows.


Map: 100%|██████████| 800/800 [00:00<00:00, 24319.39 examples/s]
Traceback (most recent call last):
  File "/kaggle/working/neuro-symbolic-pathfinding/train_sft.py", line 181, in <module>
    main()
  File "/kaggle/working/neuro-symbolic-pathfinding/train_sft.py", line 149, in main
    training_args = SFTConfig(
                    ^^^^^^^^^^
TypeError: SFTConfig.__init__() got an unexpected keyword argument 'max_seq_length'



❌ FAILED (exit code 1, 0.5 min) -- see output above


{'stage': 'SFT: mixed-format (Gemma 4 E2B)',
 'status': 'failed',
 'exit_code': 1,
 'elapsed_min': 0.5}

In [19]:
GRPO_MIXED_TIMING_DIR = "./results/grpo_gemma4-e2b_mixed_timing"
run_stage("GRPO timing test: mixed-format",
          ["train_grpo.py", "--model", "gemma4-e2b", "--adapter_path", SFT_MIXED_DIR,
           "--condition", "mixed", "--grid_size", "5", "--n_tasks", "20", "--max_steps", "20",
           "--output_dir", GRPO_MIXED_TIMING_DIR],
          required_paths=[SFT_MIXED_DIR], timeout=TIMEOUT_QUICK)



▶ GRPO timing test: mixed-format
⏭  SKIPPED -- required input not found: ./results/sft_gemma4-e2b_mixed


{'stage': 'GRPO timing test: mixed-format',
 'status': 'skipped',
 'reason': 'missing ./results/sft_gemma4-e2b_mixed'}

In [20]:
GRPO_MIXED_DIR = "./results/grpo_gemma4-e2b_mixed"
mixed_steps = steps_for_budget(GRPO_MIXED_TIMING_DIR)
run_stage(f"GRPO: mixed-format full run ({mixed_steps} steps)",
          ["train_grpo.py", "--model", "gemma4-e2b", "--adapter_path", SFT_MIXED_DIR,
           "--condition", "mixed", "--grid_size", "5", "--n_tasks", "50",
           "--max_steps", str(mixed_steps), "--output_dir", GRPO_MIXED_DIR],
          required_paths=[SFT_MIXED_DIR], timeout=TIMEOUT_GRPO_FULL)


  (no timing.json at ./results/grpo_gemma4-e2b_mixed_timing, using default 200 steps)

▶ GRPO: mixed-format full run (200 steps)
⏭  SKIPPED -- required input not found: ./results/sft_gemma4-e2b_mixed


{'stage': 'GRPO: mixed-format full run (200 steps)',
 'status': 'skipped',
 'reason': 'missing ./results/sft_gemma4-e2b_mixed'}

In [21]:
run_stage("Eval mixed-format checkpoint: MazeBench",
          ["eval.py", "--model", "gemma4-e2b", "--checkpoint", GRPO_MIXED_DIR,
           "--benchmark", "mazebench", "--n", "20", "--output_dir", "./results/eval"],
          required_paths=[GRPO_MIXED_DIR], timeout=TIMEOUT_EVAL)
run_stage("Eval mixed-format checkpoint: GridRoute NL",
          ["eval.py", "--model", "gemma4-e2b", "--checkpoint", GRPO_MIXED_DIR,
           "--benchmark", "gridroute-nl", "--grid_size", "5", "--n", "50", "--output_dir", "./results/eval"],
          required_paths=[GRPO_MIXED_DIR], timeout=TIMEOUT_EVAL)



▶ Eval mixed-format checkpoint: MazeBench
⏭  SKIPPED -- required input not found: ./results/grpo_gemma4-e2b_mixed

▶ Eval mixed-format checkpoint: GridRoute NL
⏭  SKIPPED -- required input not found: ./results/grpo_gemma4-e2b_mixed


{'stage': 'Eval mixed-format checkpoint: GridRoute NL',
 'status': 'skipped',
 'reason': 'missing ./results/grpo_gemma4-e2b_mixed'}

## Phase 2, recipe 3: consistency-reward (one candidate recipe, not this project's headline claim)

Adapts Elhady et al.'s cross-lingual consistency-reward mechanism to cross-format spatial reasoning -- try it, report honestly whether it beats `mixed` or not. See `train_grpo.py`'s `make_reward_fn` docstring for the exact mechanism. This condition generates an extra partner completion per reward call (roughly double the per-step cost of single/mixed) -- `steps_for_budget()` accounts for that automatically since it reads THIS condition's own timing test, not single/mixed's.

In [22]:
SFT_CONSISTENCY_DIR = SFT_MIXED_DIR  # consistency reuses the same mixed-format SFT warm-start
GRPO_CONSISTENCY_TIMING_DIR = "./results/grpo_gemma4-e2b_consistency_timing"
run_stage("GRPO timing test: consistency-reward",
          ["train_grpo.py", "--model", "gemma4-e2b", "--adapter_path", SFT_CONSISTENCY_DIR,
           "--condition", "consistency", "--grid_size", "5", "--n_tasks", "20", "--max_steps", "20",
           "--output_dir", GRPO_CONSISTENCY_TIMING_DIR],
          required_paths=[SFT_CONSISTENCY_DIR], timeout=TIMEOUT_QUICK)



▶ GRPO timing test: consistency-reward
⏭  SKIPPED -- required input not found: ./results/sft_gemma4-e2b_mixed


{'stage': 'GRPO timing test: consistency-reward',
 'status': 'skipped',
 'reason': 'missing ./results/sft_gemma4-e2b_mixed'}

In [23]:
GRPO_CONSISTENCY_DIR = "./results/grpo_gemma4-e2b_consistency"
consistency_steps = steps_for_budget(GRPO_CONSISTENCY_TIMING_DIR)
run_stage(f"GRPO: consistency-reward full run ({consistency_steps} steps)",
          ["train_grpo.py", "--model", "gemma4-e2b", "--adapter_path", SFT_CONSISTENCY_DIR,
           "--condition", "consistency", "--grid_size", "5", "--n_tasks", "50",
           "--max_steps", str(consistency_steps), "--output_dir", GRPO_CONSISTENCY_DIR],
          required_paths=[SFT_CONSISTENCY_DIR], timeout=TIMEOUT_GRPO_FULL)


  (no timing.json at ./results/grpo_gemma4-e2b_consistency_timing, using default 200 steps)

▶ GRPO: consistency-reward full run (200 steps)
⏭  SKIPPED -- required input not found: ./results/sft_gemma4-e2b_mixed


{'stage': 'GRPO: consistency-reward full run (200 steps)',
 'status': 'skipped',
 'reason': 'missing ./results/sft_gemma4-e2b_mixed'}

In [24]:
run_stage("Eval consistency-reward checkpoint: MazeBench",
          ["eval.py", "--model", "gemma4-e2b", "--checkpoint", GRPO_CONSISTENCY_DIR,
           "--benchmark", "mazebench", "--n", "20", "--output_dir", "./results/eval"],
          required_paths=[GRPO_CONSISTENCY_DIR], timeout=TIMEOUT_EVAL)
run_stage("Eval consistency-reward checkpoint: GridRoute NL",
          ["eval.py", "--model", "gemma4-e2b", "--checkpoint", GRPO_CONSISTENCY_DIR,
           "--benchmark", "gridroute-nl", "--grid_size", "5", "--n", "50", "--output_dir", "./results/eval"],
          required_paths=[GRPO_CONSISTENCY_DIR], timeout=TIMEOUT_EVAL)



▶ Eval consistency-reward checkpoint: MazeBench
⏭  SKIPPED -- required input not found: ./results/grpo_gemma4-e2b_consistency

▶ Eval consistency-reward checkpoint: GridRoute NL
⏭  SKIPPED -- required input not found: ./results/grpo_gemma4-e2b_consistency


{'stage': 'Eval consistency-reward checkpoint: GridRoute NL',
 'status': 'skipped',
 'reason': 'missing ./results/grpo_gemma4-e2b_consistency'}

## Run summary

At-a-glance status of every stage attempted -- read this first when checking back on an unattended run, before digging into the full cell outputs above.

In [25]:
import pandas as pd

summary_df = pd.DataFrame(stage_results)
if not summary_df.empty:
    ok = (summary_df["status"] == "ok").sum()
    print(f"{ok}/{len(summary_df)} stages completed successfully.\n")
pd.set_option("display.max_colwidth", None)
summary_df


3/25 stages completed successfully.



,stage,status,elapsed_min,exit_code,reason
0,Feasibility check (all candidate models),ok,4.0,NaN,NaN
1,"AlphaMaze baseline: MazeBench (full precision, faithful replication)",ok,18.3,NaN,NaN
2,AlphaMaze baseline: GridRoute NL,ok,24.0,NaN,NaN
3,Gemma 4 E2B baseline: MazeBench,timeout,60.0,NaN,NaN
4,Gemma 4 E2B baseline: GridRoute NL,timeout,60.0,NaN,NaN
5,Gemma 4 E4B baseline: MazeBench,timeout,60.0,NaN,NaN
6,Gemma 4 E4B baseline: GridRoute NL,timeout,60.0,NaN,NaN
7,[full-precision] Gemma 4 E2B baseline: MazeBench,timeout,60.0,NaN,NaN
8,[full-precision] Gemma 4 E2B baseline: GridRoute NL,timeout,60.0,NaN,NaN
9,[full-precision] Gemma 4 E4B baseline: MazeBench,failed,0.6,1.0,NaN


## Aggregate benchmark results into one comparison table

In [26]:
import glob

rows = []
for path in sorted(glob.glob("./results/eval/*.json")):
    try:
        with open(path) as f:
            d = json.load(f)
    except (json.JSONDecodeError, OSError) as e:
        print(f"Skipping unreadable results file {path}: {e}")
        continue
    label = path.split("/")[-1]
    row = {"file": label, "model": d.get("model"), "checkpoint": d.get("checkpoint") or "(none)",
           "benchmark": d.get("benchmark"), "n": d.get("n"),
           "4bit": d.get("load_in_4bit")}  # AlphaMaze rows are always False (enforced in eval.py);
                                            # False Gemma 4 rows are the Phase 1b ablation
    if "mazebench" in d.get("benchmark", ""):
        row["score"] = d.get("accuracy")
        row["used_official_scoring"] = d.get("used_official_scoring")
    else:
        row["valid_rate"] = d.get("valid_rate")
        row["optimal_rate"] = d.get("optimal_rate")
    rows.append(row)

results_df = pd.DataFrame(rows)
if not results_df.empty:
    results_df.to_csv("./results/comparison_table.csv", index=False)
else:
    print("No eval result files found yet in ./results/eval/ -- nothing to aggregate.")
results_df


,file,model,checkpoint,benchmark,n,4bit,valid_rate,optimal_rate,score,used_official_scoring
0,alphamaze_gridroute-nl_20260715_194101.json,alphamaze,(none),gridroute-nl-5x5,50,False,0.16,0.14,NaN,NaN
1,alphamaze_mazebench_20260715_191704.json,alphamaze,(none),mazebench,30,False,NaN,NaN,0.833333,True


## Download results

In [27]:
import shutil
shutil.make_archive("/kaggle/working/results", "zip", "./results")
print("Saved: /kaggle/working/results.zip -- download it from the Kaggle output panel on the right.")
print("Includes stage_log.json (this run's full stage-by-stage status) and comparison_table.csv.")


Saved: /kaggle/working/results.zip -- download it from the Kaggle output panel on the right.
Includes stage_log.json (this run's full stage-by-stage status) and comparison_table.csv.
